In [1]:
import os
import subprocess

# Directorio de destino en tu computadora (ajusta la ruta si es necesario)
target_dir = '/ruta/a/tu/carpeta/my_extjwnl_mcr30_spa'
os.makedirs(target_dir, exist_ok=True)

print("Carpeta creada (o ya existente) en:", target_dir)

# URL del repositorio y ruta local
repo_url = 'https://github.com/extjwnl/extjwnl-data-mcr30.git'
repo_local_path = './extjwnl-data-mcr30'  # Aquí usa la ruta donde quieras clonar (actualmente en el directorio actual)

# Clonamos si no existe
if not os.path.exists(repo_local_path):
    subprocess.run(['git', 'clone', repo_url, repo_local_path], check=True)
else:
    print("Ya existe la carpeta extjwnl-data-mcr30.")

# Ruta relativa al español (SPA)
spa_relative_path = 'lang-spa/src/main/resources/net/sf/extjwnl/data/mcr30/spa'
spa_full_path = os.path.join(repo_local_path, spa_relative_path)

print("Contenido en la carpeta de español (SPA):")
for item in os.listdir(spa_full_path):
    print(item)


Carpeta creada (o ya existente) en: /ruta/a/tu/carpeta/my_extjwnl_mcr30_spa
Contenido en la carpeta de español (SPA):
adj.exc
adv.exc
cntlist
cntlist.rev
data.adj
data.adv
data.noun
data.verb
ili.csv
index.adj
index.adv
index.noun
index.sense
index.verb
lexnames
noun.exc
res_properties.xml
verb.exc


In [7]:
index_sense_path = "C:\\Users\\danhm\\Desktop\\SensekysESP\\extjwnl-data-mcr30\\lang-spa\\src\\main\\resources\\net\\sf\\extjwnl\\data\\mcr30\\spa\\index.sense"

In [10]:
import requests
import csv
import time

# --------------------------
# Configuración
# --------------------------
index_sense_path = "C:\\Users\\danhm\\Desktop\\SensekysESP\\extjwnl-data-mcr30\\lang-spa\\src\\main\\resources\\net\\sf\\extjwnl\\data\\mcr30\\spa\\index.sense"   # Cambia por la ruta real
output_csv_path = "mapeo_sensekeys_arasaac.csv"
api_base_url = "https://api.arasaac.org/v1"

# Mapear dígitos POS a letra POS (WordNet)
pos_map = {
    "1": "n",  # noun
    "2": "v",  # verb
    "3": "a",  # adjective
    "4": "r"   # adverb
}

# --------------------------
# Función para consultar ARASAAC
# --------------------------
def get_pictogram_id(word):
    # Reemplazar guiones bajos por espacios
    search_word = word.replace('_', ' ')
    url = f"{api_base_url}/pictograms/es/search/{search_word}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            if data:
                return data[0]['_id']  # Primer pictograma encontrado
    except Exception as e:
        print(f"Error buscando {word}: {e}")
    return None

# --------------------------
# Procesar index.sense
# --------------------------
with open(index_sense_path, "r", encoding="utf-8") as f_in, \
     open(output_csv_path, "w", encoding="utf-8", newline="") as f_out:

    writer = csv.writer(f_out)
    writer.writerow(["word", "pictogram_id", "synset", "word_senses"])

    processed_words = set()  # Para evitar consultas repetidas

    for line in f_in:
        line = line.strip()
        if not line or line.startswith('#'):
            continue

        parts = line.split()
        sense_key = parts[0]          # ej. pavimento%1:06:01::
        synset_offset = parts[1]      # ej. 04215402

        lemma = sense_key.split('%')[0]
        pos_digit = sense_key.split('%')[1].split(':')[0]
        pos_letter = pos_map.get(pos_digit, "n")
        synset_str = f"{synset_offset}-{pos_letter}"
        word_senses = sense_key

        # Solo consultamos una vez por palabra
        if lemma not in processed_words:
            pictogram_id = get_pictogram_id(lemma)
            processed_words.add(lemma)
            time.sleep(0.3)  # Para no saturar la API (ajusta si es necesario)
        else:
            pictogram_id = None  # Ya lo buscaste antes (puedes guardar si quieres)

        # Escribir en CSV
        writer.writerow([lemma, pictogram_id, synset_str, word_senses])

print(f"CSV generado: {output_csv_path}")


IndexError: list index out of range